In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
TABLES_DIR = os.path.join(OUTPUTS_DIR, "tables")
FIGURES_DIR = os.path.join(OUTPUTS_DIR, "figures")

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

FAKE_LABEL = 1
REAL_LABEL = 0
LABEL_ORDER = [FAKE_LABEL, REAL_LABEL]
LABEL_TICKS = ["Fake(1)", "Real(0)"]

train_path = os.path.join(DATA_PROCESSED_DIR, "train.csv")
val_path = os.path.join(DATA_PROCESSED_DIR, "val.csv")
test_path = os.path.join(DATA_PROCESSED_DIR, "test.csv")

train_df = pd.read_csv(train_path, usecols=["full_text", "label"])
val_df = pd.read_csv(val_path, usecols=["full_text", "label"])
test_df = pd.read_csv(test_path, usecols=["full_text", "label"])

X_train = train_df["full_text"].astype(str).tolist()
y_train = train_df["label"].astype(int).to_numpy()

X_val = val_df["full_text"].astype(str).tolist()
y_val = val_df["label"].astype(int).to_numpy()

X_test = test_df["full_text"].astype(str).tolist()
y_test = test_df["label"].astype(int).to_numpy()

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

models = {
    "TFIDF_LogReg": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="saga"
    ),
    "TFIDF_LinearSVC": LinearSVC(
        class_weight="balanced"
    )
}

def eval_model(name, model, X, y):
    y_pred = model.predict(X)
    acc = accuracy_score(y, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y, y_pred,
        average="binary",
        pos_label=FAKE_LABEL,
        zero_division=0
    )
    return {
        "model": name,
        "accuracy": acc,
        "precision_fake(1)": p,
        "recall_fake(1)": r,
        "f1_fake(1)": f1
    }, y_pred

results = []

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    row_val, _ = eval_model(name, model, X_val_tfidf, y_val)
    results.append({**row_val, "split": "val"})

results_df = pd.DataFrame(results).sort_values(["split", "f1_fake(1)"], ascending=[True, False])
print(results_df)

best_name = results_df.sort_values("f1_fake(1)", ascending=False).iloc[0]["model"]
best_model = models[best_name]
best_model.fit(X_train_tfidf, y_train)

test_row, y_test_pred = eval_model(best_name, best_model, X_test_tfidf, y_test)
test_row["split"] = "test"
print("\nBest on val:", best_name)
print("\nTest metrics:")
print(test_row)
print("\nClassification report (test):")
print(classification_report(y_test, y_test_pred, labels=LABEL_ORDER, target_names=LABEL_TICKS, digits=4))

metrics_out = pd.concat([results_df, pd.DataFrame([test_row])], ignore_index=True)
metrics_path = os.path.join(TABLES_DIR, "tfidf_baselines_metrics.csv")
metrics_out.to_csv(metrics_path, index=False)
print("\nSaved metrics to:", metrics_path)

cm = confusion_matrix(y_test, y_test_pred, labels=LABEL_ORDER)
fig = plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest")
plt.title(f"Confusion Matrix (Test) - {best_name}")
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, LABEL_TICKS)
plt.yticks(tick_marks, LABEL_TICKS)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], "d"), ha="center", va="center")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()

cm_path = os.path.join(FIGURES_DIR, f"confusion_matrix_{best_name}.png")
plt.savefig(cm_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved confusion matrix to:", cm_path)


In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
TABLES_DIR = os.path.join(OUTPUTS_DIR, "tables")

os.makedirs(TABLES_DIR, exist_ok=True)

FAKE_LABEL = 1
REAL_LABEL = 0
LABEL_ORDER = [FAKE_LABEL, REAL_LABEL]
LABEL_TICKS = ["Fake(1)", "Real(0)"]

train_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "train.csv"), usecols=["full_text", "label"])
val_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "val.csv"), usecols=["full_text", "label"])
test_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "test.csv"), usecols=["full_text", "label"])

X_train = train_df["full_text"].astype(str).tolist()
y_train = train_df["label"].astype(int).to_numpy()

X_val = val_df["full_text"].astype(str).tolist()
y_val = val_df["label"].astype(int).to_numpy()

X_test = test_df["full_text"].astype(str).tolist()
y_test = test_df["label"].astype(int).to_numpy()

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

models = {
    "TFIDF_LogReg": LogisticRegression(max_iter=2000, class_weight="balanced", solver="saga"),
    "TFIDF_LinearSVC": LinearSVC(class_weight="balanced"),
    "TFIDF_MultinomialNB": MultinomialNB(),
    "TFIDF_SGDLinear": SGDClassifier(loss="hinge", max_iter=2000, class_weight="balanced", random_state=42)
}

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    p_fake, r_fake, f1_fake, _ = precision_recall_fscore_support(
        y_true, y_pred,
        average="binary",
        pos_label=FAKE_LABEL,
        zero_division=0
    )
    return acc, f1_macro, f1_w, p_fake, r_fake, f1_fake

rows = []

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_val_pred = model.predict(X_val_tfidf)
    acc, f1_macro, f1_w, p_fake, r_fake, f1_fake = compute_metrics(y_val, y_val_pred)
    rows.append({
        "model": name,
        "split": "val",
        "accuracy": acc,
        "macro_f1": f1_macro,
        "weighted_f1": f1_w,
        "precision_fake(1)": p_fake,
        "recall_fake(1)": r_fake,
        "f1_fake(1)": f1_fake
    })

results_df = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
print(results_df)

best_name = results_df.iloc[0]["model"]
best_model = models[best_name]
best_model.fit(X_train_tfidf, y_train)

y_test_pred = best_model.predict(X_test_tfidf)
acc, f1_macro, f1_w, p_fake, r_fake, f1_fake = compute_metrics(y_test, y_test_pred)

test_row = pd.DataFrame([{
    "model": best_name,
    "split": "test",
    "accuracy": acc,
    "macro_f1": f1_macro,
    "weighted_f1": f1_w,
    "precision_fake(1)": p_fake,
    "recall_fake(1)": r_fake,
    "f1_fake(1)": f1_fake
}])

print("\nBest model on val (by macro_f1):", best_name)
print("\nTest classification report:")
print(classification_report(y_test, y_test_pred, labels=LABEL_ORDER, target_names=LABEL_TICKS, digits=4))

out_df = pd.concat([results_df, test_row], ignore_index=True)
out_path = os.path.join(TABLES_DIR, "tfidf_baselines_metrics_v2.csv")
out_df.to_csv(out_path, index=False)
print("\nSaved:", out_path)